# 第 17 节: TRPO (Trust Region Policy Optimization)

## 本节位置
Importance Sampling (16) -> TRPO (17) -> GAE (18) -> PPO (19)

## 学习目标
1. 理解信任域方法的动机
2. 掌握 TRPO surrogate objective 推导
3. 理解 KL divergence 如何约束策略更新
4. 实现 TRPO 的关键组件 (KL, HVP, CG, Line Search)
5. 理解 TRPO 与 PPO 的关系

## 1. 策略更新的核心困境

策略梯度方法的核心问题是步长选择:
- 步长太小: 收敛慢
- 步长太大: 策略崩溃 (performance collapse)

原因: 参数空间的距离不等于策略空间的距离。
小参数变化可能导致策略分布的剧烈变化。

$$|\theta_1 - \theta_2|_2 \ll 1 \quad\not\Rightarrow\quad \pi_{\theta_1} \approx \pi_{\theta_2}$$

## 2. Surrogate Objective

使用重要性采样, TRPO 优化以下替代目标:

$$L_{\pi_{\text{old}}}(\pi_\theta) = \mathbb{E}_{a \sim \pi_{\text{old}}}\left[ \frac{\pi_\theta(a|s)}{\pi_{\text{old}}(a|s)} A^{\pi_{\text{old}}}(s, a) \right]$$

约束条件:
$$\mathbb{E}_s[D_{KL}(\pi_{\text{old}}(\cdot|s) \| \pi_\theta(\cdot|s))] \leq \delta$$

**重要**: $L_{\pi_{\text{old}}}$ 是 surrogate (替代) 目标, 不等于真正的 $J(\pi_\theta)$,
因为它忽略了新策略导致的状态访问分布变化。TRPO 理论上有 monotonic improvement 保证,
但实际算法对理论过程做了多项近似, 原论文表述为 tend to produce monotonic improvements。

## 3. 完整 TRPO 算法流程

```
1. 用当前策略 pi_old 收集轨迹数据
2. 计算 advantage estimates A_t
3. 计算策略梯度 g = grad_theta L(theta)
4. 用 conjugate gradient 求解 F^{-1} g (natural gradient)
5. 计算步长 beta = sqrt(2*delta / (g^T * F^{-1} * g))
6. 候选参数: theta = theta_old + beta * F^{-1} * g
7. Backtracking line search: 从最大步长开始指数衰减
   - 检查 KL constraint: D_KL(pi_old || pi_new) <= delta
   - 检查 surrogate objective 是否改进
8. 如果满足约束则接受更新, 否则缩小步长重试
```

下面分别演示 TRPO 的关键组件。完整 TRPO 需要组合所有这些组件,
实现复杂度高。PPO 用更简单的 clipping 机制达到了类似效果, 是实践中的首选。

### 组件 1: KL Divergence 的正确计算

**关键**: 必须先保存旧策略的 logits (clone), 更新后再计算 KL。
常见错误是用 logits.detach() vs logits (更新前两者数值相同, KL 恒为 0)。

In [ ]:
import sys; sys.path.insert(0, '/workspace/data/vggt-omega/rl')
from rl_course.utils.seeding import set_seed; set_seed(42)
import numpy as np
import torch; import torch.nn as nn; import torch.nn.functional as F
import torch.optim as optim
import matplotlib; matplotlib.use('Agg')
import matplotlib.pyplot as plt
import os; FIG_DIR = 'outputs/figures'; os.makedirs(FIG_DIR, exist_ok=True)

# 创建简单策略网络
class TinyPolicy(nn.Module):
    def __init__(self, n_actions=3):
        super().__init__()
        self.fc = nn.Linear(4, n_actions)

    def forward(self, x):
        return self.fc(x)

def compute_kl(old_logits, new_logits):
    old_probs = torch.softmax(old_logits, dim=-1)
    old_logp = torch.log_softmax(old_logits, dim=-1)
    new_logp = torch.log_softmax(new_logits, dim=-1)
    return (old_probs * (old_logp - new_logp)).sum(dim=-1).mean()

# 演示: 正确的 KL 计算
policy = TinyPolicy()
x = torch.randn(32, 4)

# 步骤 1: 保存旧 logits (clone!)
with torch.no_grad():
    old_logits = policy(x).clone()

# 步骤 2: 做梯度更新 (模拟策略变化)
optimizer = optim.SGD(policy.parameters(), lr=0.1)
loss = policy(x).mean()
optimizer.zero_grad(); loss.backward(); optimizer.step()

# 步骤 3: 计算新 logits
with torch.no_grad():
    new_logits = policy(x)

# 步骤 4: 计算真实的 KL (非零!)
kl = compute_kl(old_logits, new_logits)
print(f"KL(pi_old || pi_new) = {kl.item():.6f}")
print("正确! 因为 old_logits 是更新前 clone() 保存的, new_logits 是更新后的。")

### 组件 2: Fisher-Vector Product (HVP)

Fisher Information Matrix F = E[nabla log pi * nabla log pi^T]。
F * v 可通过两次 autograd.grad 计算, 无需显式构建和求逆 F。

In [ ]:
def fisher_vector_product(policy, states, vector, damping=0.1):
    # Step 1: KL 散度的梯度 (create_graph=True 保留计算图)
    logits = policy(states)
    log_probs = torch.log_softmax(logits, dim=-1)
    grads = torch.autograd.grad(
        log_probs.mean(), policy.parameters(),
        create_graph=True
    )
    # Step 2: 与 vector 做内积, 再求一次梯度
    grad_dot_v = sum((g * v).sum() for g, v in zip(grads, vector))
    fvp = torch.autograd.grad(grad_dot_v, policy.parameters())
    # Step 3: 加 damping 保证正定性
    return [f + damping * v for f, v in zip(fvp, vector)]

# 演示
x_test = torch.randn(16, 4)
v = [torch.randn_like(p) for p in policy.parameters()]
fvp = fisher_vector_product(policy, x_test, v)
print("Fisher-vector product 计算完成:")
for i, f in enumerate(fvp):
    print(f"  Param {i}: Fv norm = {f.norm().item():.6f}")
print("Fvp 是 natural gradient 计算的核心, 避免显式构建 F")

### 组件 3: Conjugate Gradient 求解 F^{-1}g

共轭梯度法只使用 F-vector product, 不显式构建 F, 迭代求解 Fx = g。

In [ ]:
def conjugate_gradient(fvp_func, g, n_steps=10, tol=1e-10):
    # 展平所有参数
    def flatten(tensors):
        return torch.cat([t.reshape(-1) for t in tensors])
    def unflatten(flat, shapes):
        result, idx = [], 0
        for s in shapes:
            n = s.numel()
            result.append(flat[idx:idx+n].reshape(s))
            idx += n
        return result

    shapes = [p.shape for p in g]
    g_flat, x_flat = flatten(g), torch.zeros_like(flatten(g))
    r, p_vec = g_flat.clone(), g_flat.clone()
    r_dot_r = r.dot(r)
    residuals = []

    for i in range(n_steps):
        fp_vec = fvp_func(unflatten(p_vec, shapes))
        fp_flat = flatten(fp_vec)
        alpha = r_dot_r / (p_vec.dot(fp_flat) + 1e-10)
        x_flat += alpha * p_vec
        r -= alpha * fp_flat
        new_r_dot_r = r.dot(r)
        residuals.append(new_r_dot_r.item())
        if new_r_dot_r < tol: break
        beta = new_r_dot_r / r_dot_r
        p_vec = r + beta * p_vec
        r_dot_r = new_r_dot_r

    print(f"CG converged at iteration {i+1}, residual: {new_r_dot_r:.2e}")
    return unflatten(x_flat, shapes), residuals

# 演示
g = [torch.randn_like(p) for p in policy.parameters()]

def fvp_wrapper(v):
    return fisher_vector_product(policy, x_test, v)

nat_grad, residuals = conjugate_gradient(fvp_wrapper, g, n_steps=15)

print("Regular vs Natural gradient direction:")
for gi, ngi in zip(g, nat_grad):
    cos = (gi.reshape(-1).dot(ngi.reshape(-1))) / (gi.norm() * ngi.norm() + 1e-8)
    print(f"  cos_sim={cos:.4f}")

# CG 收敛曲线
fig, ax = plt.subplots(figsize=(8, 3))
ax.semilogy(residuals, 'o-', linewidth=1.5)
ax.set_xlabel('CG Iteration'); ax.set_ylabel('Residual (r^T r)')
ax.set_title('Conjugate Gradient Convergence'); ax.grid(True, alpha=0.3)
plt.savefig(f'{FIG_DIR}/17_cg_convergence.png', dpi=100); plt.close()
print("CG convergence plot saved")

### 组件 4: Backtracking Line Search

给定搜索方向, 从最大步长开始指数衰减, 找到满足 KL 约束的步长。

In [ ]:
def backtracking_line_search(policy, old_params, search_dir, old_logits,
                              states, max_kl, max_backtracks=10):
    step_size, decay = 1.0, 0.8
    for i in range(max_backtracks):
        with torch.no_grad():
            for p, p_old, d in zip(policy.parameters(), old_params, search_dir):
                p.copy_(p_old + step_size * d)
        with torch.no_grad():
            kl = compute_kl(old_logits, policy(states))
        if kl <= max_kl:
            return True, step_size, kl.item()
        step_size *= decay

    # Restore old params
    with torch.no_grad():
        for p, p_old in zip(policy.parameters(), old_params):
            p.copy_(p_old)
    return False, 0.0, kl.item()

# 演示
old_params = [p.clone() for p in policy.parameters()]
with torch.no_grad():
    old_logits_demo = policy(x_test).clone()

search_dir = [torch.randn_like(p) * 0.01 for p in policy.parameters()]
accepted, step, kl = backtracking_line_search(
    policy, old_params, search_dir, old_logits_demo, x_test, max_kl=0.01
)
print(f"Line search: accepted={accepted}, final_step={step:.4f}, KL={kl:.6f}")

# 可视化: 步长 vs KL
steps = [1.0 * (0.8**i) for i in range(10)]
kls = []
with torch.no_grad():
    for s in steps:
        for p, p_old, d in zip(policy.parameters(), old_params, search_dir):
            p.copy_(p_old + s * d)
        kls.append(compute_kl(old_logits_demo, policy(x_test)).item())

fig, ax = plt.subplots(figsize=(8, 3))
ax.plot(steps, kls, 'o-', linewidth=1.5)
ax.axhline(y=0.01, color='r', linestyle='--', label='max_kl=0.01')
ax.set_xlabel('Step Size'); ax.set_ylabel('KL Divergence')
ax.set_title('Backtracking Line Search'); ax.legend(); ax.grid(True, alpha=0.3)
ax.set_xscale('log'); ax.set_yscale('log')
plt.savefig(f'{FIG_DIR}/17_line_search.png', dpi=100); plt.close()
print("Line search visualization saved")

## 4. TRPO vs PPO 对比

| 组件 | TRPO | PPO |
|------|------|-----|
| 约束方式 | KL 硬约束 + line search | Clipping (软约束) |
| 优化方法 | Natural gradient (CG + HVP) | Adam SGD |
| 每步计算 | O(k * params) (k=CG iterations) | O(params) |
| 理论保证 | 单调改进 (近似) | 无严格保证 |
| 实现复杂度 | 高 | 低 |
| 实际使用 | 较少 | 非常广泛 |

完整的 TRPO 实现需要组合以上所有组件。PPO 用更简单的 clipping 机制
达到了类似效果, 是绝大多数实际应用的首选。
本节的目标是理解 TRPO 的核心组件, 而非生成一个可训练的 TRPO。

## 5. 本节总结

1. TRPO 通过 KL 约束限制了策略更新的幅度
2. 核心工具: Surrogate objective + KL constraint + Natural gradient
3. 实现需要: Fisher-vector product, Conjugate gradient, Line search
4. TRPO 有更好的理论基础, 但 PPO 在实践中更实用
5. 理解 TRPO 的组件有助于理解 PPO 的设计动机

## 6. 练习
1. 修改 max_kl (0.001, 0.01, 0.1), 观察 line search 接受多少步长
2. 比较 CG 解 F^{-1}g 与普通梯度 g 的方向差异 (cosine similarity)
3. 为什么 Fvp 需要两次 autograd.grad?
4. 阅读 TRPO 论文的 monotonic improvement 证明部分

---

*下一节: [18_gae.ipynb](18_gae.ipynb) — GAE 推导与实验*